### Concatenar DBF

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# Definir o caminho para a pasta dos arquivos como o diretório atual
caminho_arquivos = os.path.abspath('')  # Obtém o caminho absoluto do diretório atual

# Lista com os nomes dos arquivos
arquivos = ['den_2020.xlsx', 'den_2021.xlsx', 'den_2022.xlsx', 'den_2023.xlsx', 'den_2024.xlsx']

# Criar uma lista vazia para armazenar os DataFrames
dfs = []

# Ler cada arquivo e adicionar o DataFrame à lista
for arquivo in arquivos:
    caminho_completo = os.path.join(caminho_arquivos, arquivo)  # Combina a pasta com o nome do arquivo
    df = pd.read_excel(caminho_completo)
    dfs.append(df)

# Concatenar todos os DataFrames em um único DataFrame
df_final = pd.concat(dfs, ignore_index=True)

# Salvar o DataFrame concatenado em um novo arquivo Excel no diretório atual
output_file_path = os.path.join(caminho_arquivos, 'den_2020_2024.xlsx')
df_final.to_excel(output_file_path, index=False)

print('Arquivos concatenados com sucesso!')

### Análise Geral

In [ ]:
# Carregar o arquivo Excel
df_notificados = pd.read_excel(output_file_path)

# Contar ocorrências de casos notificados
df_notificados['ID_AGRAVO'].value_counts().reset_index(name='Total')

In [ ]:
# Total descartados
df_notificados['CLASSI_FIN'].value_counts().reset_index(name='Total')

In [ ]:
df_notificados_ano = df_notificados['NU_ANO'].value_counts().reset_index(name='Total')
df_notificados_ano

In [ ]:
df_notificados_ano = df_notificados.groupby(['NU_ANO', 'CLASSI_FIN']).size().unstack(fill_value=0)
df_notificados_ano.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Casos Notificados por Ano e Classificação Final')
plt.xlabel('Ano')
plt.ylabel('Número de Casos')
plt.legend(title='Classificação Final')
plt.tight_layout()
# Salvar o gráfico como imagem no diretório atual
grafico_path = os.path.join(caminho_arquivos, 'casos_notificados_ano.png')
plt.savefig(grafico_path)

### Filtragem de dados - Clínico Epidemiológico

In [ ]:
# Carregar o arquivo Excel
file_path = 'den_2020_2024.xlsx'
df_clepi = pd.read_excel(file_path)

# Remover colunas desnecessárias
colunas_para_remover = [
    'SOUNDEX', 'ID_DISTRIT', 'ID_GEO1', 'ID_GEO2', 'DIABETES', 'HEMATOLOG', 'HEPATOPAT', 'RENAL', 'HIPERTENSA',
    'ACIDO_PEPT', 'AUTO_IMUNE', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 
    'HOSPITALIZ', 'DT_INTERNA', 'UF', 'MUNICIPIO', 'HOSPITAL', 'DDD_HOSP', 'TEL_HOSP', 'TPAUTOCTO', 'COUFINF', 
    'COPAISINF', 'COMUNINF', 'CODISINF', 'CO_BAINF', 'NOBAIINF', 'DOENCA_TRA', 'CLINC_CHIK', 'ALRM_HIPOT', 
    'ALRM_PLAQ', 'ALRM_VOM', 'ALRM_SANG', 'ALRM_HEMAT', 'ALRM_ABDOM', 'ALRM_LETAR', 'ALRM_HEPAT', 'ALRM_LIQ', 
    'DT_ALRM', 'GRAV_PULSO', 'GRAV_CONV', 'GRAV_ENCH', 'GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 
    'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO', 
    'DT_GRAV', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 
    'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLICA', 'NU_LOTE_I', 'NDUPLIC_N', 'DT_TRANSUS', 
    'DT_TRANSDM', 'DT_TRANSSM', 'DT_TRANSRM', 'DT_TRANSRS', 'DT_TRANSSE', 'NU_LOTE_V', 'NU_LOTE_H', 'CS_FLXRET', 
    'FLXRECEBI', 'IDENT_MICR', 'MIGRADO_W', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE', 'ID_PAIS', 'ID_OCUPA_N', 
    'DT_SORO', 'DT_NS1', 'DT_VIRAL', 'DT_PCR', 'SOROTIPO', 'RESUL_VI_N', 'HISTOPA_N', 'IMUNOH_N', 'TP_SISTEMA', 'TP_NOT', 
    'CS_ESCOL_N', 'CS_GESTANT', 'DS_OBS']

df_clepi = df_clepi.drop(columns=colunas_para_remover, axis=1)

# Filtrar as linhas com classificação final 'descartado'
df_filtered_1 = df_clepi[df_clepi['CLASSI_FIN'] == 5]

# Filtrar as linhas com critério clinico-epidemiologico 
df_filtered_1 = df_filtered_1[df_filtered_1['CRITERIO'] == 2]

# Lista de colunas de sintomas
sintomas = ['FEBRE', 'MIALGIA', 'CEFALEIA', 
            'EXANTEMA', 'VOMITO', 'NAUSEA', 
            'ARTRALGIA', 'PETEQUIA_N', 'LACO', 
            'DOR_RETRO']

# Função para determinar se é caso suspeito
def is_suspect_case(row):
    # Verificar se o paciente tem febre
    if row['FEBRE'] != 1:
        return 'não'
    # Contar quantos sintomas o paciente tem
    num_sintomas = sum(row[sintomas] == 1)
    # Verificar se os exames não foram realizados (considerando células vazias ou valor 4)
    exames_nao_realizados = (
        (pd.isna(row['RESUL_SORO']) or row['RESUL_SORO'] == 4) and
        (pd.isna(row['RESUL_NS1']) or row['RESUL_NS1'] == 4) and
        (pd.isna(row['RESUL_PCR_']) or row['RESUL_PCR_'] == 4)
    )
    # Se tiver febre, pelo menos três sintomas e exames não realizados, é caso descartado sem critério
    return 'sim' if (num_sintomas >= 3) and exames_nao_realizados else 'não'

# Aplicar a função para cada linha do DataFrame
df_filtered_1['DESCARTE_SEM_CRITERIO'] = df_filtered_1.apply(is_suspect_case, axis=1)

# Filtrar os casos classificados como 'sim'
casos_suspeitos = df_filtered_1[df_filtered_1['DESCARTE_SEM_CRITERIO'] == 'sim']

# Contar o número de casos descartados por ano
descartados_por_ano = casos_suspeitos.groupby('NU_ANO').size().reset_index(name='Total_Suspeitos_Descartados')

# Filtrar os casos suspeitos que têm febre
casos_suspeitos_com_febre = casos_suspeitos[casos_suspeitos['FEBRE'] == 1]

# Contar o número de cada sintoma entre os casos suspeitos com febre (excluindo a febre)
sintomas_adicionais = [s for s in sintomas if s != 'FEBRE']
sintomas_contagem = casos_suspeitos_com_febre[sintomas_adicionais].apply(lambda x: (x == 1).sum()).reset_index()
sintomas_contagem.columns = ['Sintomas', 'Total']

# Adicionar a contagem de casos com febre
total_febre = casos_suspeitos_com_febre.shape[0]
sintomas_contagem = pd.concat([pd.DataFrame([['FEBRE', total_febre]], columns=['Sintomas', 'Total']), sintomas_contagem])

# Salvar o resultado em um novo arquivo Excel com múltiplas abas
output_file_path = 'descartados_2020_2024_clepi.xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    df_filtered_1.to_excel(writer, sheet_name='Casos_Descartados_Geral', index=False)
    descartados_por_ano.to_excel(writer, sheet_name='Resumo_Ano', index=False)
    sintomas_contagem.to_excel(writer, sheet_name='Sintomas', index=False)

print("Arquivo gerado com sucesso!")

In [ ]:
# Lendo o data frame
file_path = 'descartados_2020_2024_clepi.xlsx'
df_descartados_clepi = pd.read_excel(file_path)
display(df_descartados_clepi)

### Filtragem de dados - Laboratorial

In [ ]:
# Carregar o arquivo Excel
file_path = 'den_2020_2024.xlsx'
df_lab = pd.read_excel(file_path)

# Remover colunas desnecessárias
colunas_para_remover = [
    'SOUNDEX', 'ID_DISTRIT', 'ID_GEO1', 'ID_GEO2', 'DIABETES', 'HEMATOLOG', 'HEPATOPAT', 'RENAL', 'HIPERTENSA',
    'ACIDO_PEPT', 'AUTO_IMUNE', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 
    'HOSPITALIZ', 'DT_INTERNA', 'UF', 'MUNICIPIO', 'HOSPITAL', 'DDD_HOSP', 'TEL_HOSP', 'TPAUTOCTO', 'COUFINF', 
    'COPAISINF', 'COMUNINF', 'CODISINF', 'CO_BAINF', 'NOBAIINF', 'DOENCA_TRA', 'CLINC_CHIK', 'ALRM_HIPOT', 
    'ALRM_PLAQ', 'ALRM_VOM', 'ALRM_SANG', 'ALRM_HEMAT', 'ALRM_ABDOM', 'ALRM_LETAR', 'ALRM_HEPAT', 'ALRM_LIQ', 
    'DT_ALRM', 'GRAV_PULSO', 'GRAV_CONV', 'GRAV_ENCH', 'GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 
    'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO', 
    'DT_GRAV', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 
    'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLICA', 'NU_LOTE_I', 'NDUPLIC_N', 'DT_TRANSUS', 
    'DT_TRANSDM', 'DT_TRANSSM', 'DT_TRANSRM', 'DT_TRANSRS', 'DT_TRANSSE', 'NU_LOTE_V', 'NU_LOTE_H', 'CS_FLXRET', 
    'FLXRECEBI', 'IDENT_MICR', 'MIGRADO_W', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE', 'ID_PAIS', 'ID_OCUPA_N', 
    'DT_SORO', 'DT_NS1', 'DT_VIRAL', 'DT_PCR', 'SOROTIPO', 'RESUL_VI_N', 'HISTOPA_N', 'IMUNOH_N', 'TP_SISTEMA', 'TP_NOT', 
    'CS_ESCOL_N', 'CS_GESTANT', 'DS_OBS']

df_lab = df_lab.drop(columns=colunas_para_remover, axis=1)

# Filtrar as linhas com classificação final 'descartado'
df_filtered_2 = df_lab[df_lab['CLASSI_FIN'] == 5]

# Filtrar as linhas com critério laboratorial
df_filtered_2 = df_filtered_2[df_filtered_2['CRITERIO'] == 1]

# Lista de colunas de sintomas
sintomas = ['FEBRE', 'MIALGIA', 'CEFALEIA', 
            'EXANTEMA', 'VOMITO', 'NAUSEA', 
            'ARTRALGIA', 'PETEQUIA_N', 'LACO', 
            'DOR_RETRO']

# Função para determinar se é caso suspeito
def is_suspect_case(row):
    # Verificar se o paciente tem febre
    if row['FEBRE'] != 1:
        return 'não'
    # Contar quantos sintomas o paciente tem
    num_sintomas = sum(row[sintomas] == 1)
    # Verificar se os exames não foram realizados (considerando células vazias ou valor 4)
    exames_nao_realizados = (
        (pd.isna(row['RESUL_SORO']) or row['RESUL_SORO'] == 4) and
        (pd.isna(row['RESUL_NS1']) or row['RESUL_NS1'] == 4) and
        (pd.isna(row['RESUL_PCR_']) or row['RESUL_PCR_'] == 4)
    )
    # Se tiver febre, pelo menos três sintomas e exames não realizados, é caso descartado sem critério
    return 'sim' if (num_sintomas >= 3) and exames_nao_realizados else 'não'

# Aplicar a função para cada linha do DataFrame
df_filtered_2['DESCARTE_SEM_CRITERIO'] = df_filtered_2.apply(is_suspect_case, axis=1)

# Filtrar os casos classificados como 'sim'
casos_suspeitos = df_filtered_2[df_filtered_2['DESCARTE_SEM_CRITERIO'] == 'sim']

# Contar o número de casos descartados por ano
descartados_por_ano = casos_suspeitos.groupby('NU_ANO').size().reset_index(name='Total_Suspeitos_Descartados')

# Filtrar os casos suspeitos que têm febre
casos_suspeitos_com_febre = casos_suspeitos[casos_suspeitos['FEBRE'] == 1]

# Contar o número de cada sintoma entre os casos suspeitos com febre (excluindo a febre)
sintomas_adicionais = [s for s in sintomas if s != 'FEBRE']
sintomas_contagem = casos_suspeitos_com_febre[sintomas_adicionais].apply(lambda x: (x == 1).sum()).reset_index()
sintomas_contagem.columns = ['Sintomas', 'Total']

# Adicionar a contagem de casos com febre
total_febre = casos_suspeitos_com_febre.shape[0]
sintomas_contagem = pd.concat([pd.DataFrame([['FEBRE', total_febre]], columns=['Sintomas', 'Total']), sintomas_contagem])

# Contar o número de exames realizados por tipo e resultado
exames = ['RESUL_SORO', 'RESUL_NS1', 'RESUL_PCR_']
exames_contagem = pd.concat([df_filtered_2[exame].value_counts() for exame in exames], axis=1, keys=exames).fillna(0).astype(int)

# Salvar o resultado em um novo arquivo Excel com múltiplas abas
output_file_path = 'descartados_2020_2024_lab.xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    df_filtered_2.to_excel(writer, sheet_name='Casos_Descartados_Geral', index=False)
    descartados_por_ano.to_excel(writer, sheet_name='Resumo_Ano', index=False)
    sintomas_contagem.to_excel(writer, sheet_name='Sintomas', index=False)
    exames_contagem.to_excel(writer, sheet_name='Tipo_Exames', index=True)

print("Arquivo gerado com sucesso!")

In [ ]:
# Lendo o data frame
file_path = 'descartados_2020_2024_lab.xlsx'
df_descartados_lab = pd.read_excel(file_path)
display(df_descartados_lab)

### Concatenar arquivos

In [ ]:
# Definir o caminho para a pasta dos arquivos como o diretório atual
caminho_arquivos = os.path.abspath('')  # Obtém o caminho absoluto do diretório atual

# Lista com os nomes dos arquivos
arquivos = ['descartados_2020_2024_lab.xlsx', 'descartados_2020_2024_clepi.xlsx']

# Criar uma lista vazia para armazenar os DataFrames
dfs = []

# Ler cada arquivo e adicionar o DataFrame à lista
for arquivo in arquivos:
    caminho_completo = os.path.join(caminho_arquivos, arquivo)  # Combina a pasta com o nome do arquivo
    df = pd.read_excel(caminho_completo)
    dfs.append(df)

# Concatenar todos os DataFrames em um único DataFrame
df_final = pd.concat(dfs, ignore_index=True)

# Salvar o DataFrame concatenado em um novo arquivo Excel no diretório atual
output_file_path = os.path.join(caminho_arquivos, 'descartados_2020_2024_total.xlsx')
df_final.to_excel(output_file_path, index=False)

print('Arquivos concatenados com sucesso!')

### Manipulação dos dados

In [ ]:
# Carregar o arquivo Excel
file_path = 'descartados_2020_2024_total.xlsx'
den_descartados = pd.read_excel(file_path)

display(den_descartados)

In [ ]:
# Total de casos descartados
den_descartados['CLASSI_FIN'].value_counts().reset_index(name='Total')

In [ ]:
# Casos descartados, de acordo com o ano
den_descartados['NU_ANO'].value_counts().reset_index(name='Total')

In [ ]:
# Casos descartados por ano com avaliação do critério de descarte
df = pd.DataFrame(den_descartados)

# Agrupar por múltiplas colunas e contar o número de ocorrências
agrupados = df.groupby(['NU_ANO', 'CRITERIO']).size().reset_index(name='Total')

# Exibir o DataFrame agrupado
display(agrupados)

# Salvar o DataFrame agrupado em um novo arquivo Excel
output_file_path = os.path.join(caminho_arquivos, 'den_descartados_criterios.xlsx')
agrupados.to_excel(output_file_path, index=False)

In [ ]:
# Assumindo que 'agrupados' é seu DataFrame com os dados
# Criando o gráfico de barras agrupadas
ax = agrupados.unstack().plot.bar(
    color=['#1f77b4', '#ff7f0e'],  # Azul e laranja (cores mais profissionais)
    edgecolor='black',
    linewidth=0.8,
    figsize=(30, 10),
    width=0.8
)

# Configurações dos eixos
plt.xlabel('Ano', fontsize=13, fontweight='bold')
plt.ylabel('Número de Casos', fontsize=13, fontweight='bold')
plt.title('Casos de Dengue Descartados por Ano e Critério de Descarte', 
          fontsize=14, fontweight='bold', pad=20)

# Rotação e formatação dos rótulos do eixo X
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)

# Legenda personalizada
plt.legend(
    title='Critério de Descarte',
    labels=['Laboratorial (1)', 'Clínico-Epidemiológico (2)'],
    loc='upper left',
    fontsize=11,
    title_fontsize=12,
    frameon=True,
    shadow=True
)

# Grade para facilitar leitura
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Ajuste de layout
plt.tight_layout()
plt.show()

In [ ]:
# Casos descartados por ano com avaliação do critério de descarte
df = pd.DataFrame(den_descartados)

# Agrupar por múltiplas colunas e contar o número de ocorrências
agrupados = df.groupby(['NU_ANO', 'CLASSI_FIN', 'DESCARTE_SEM_CRITERIO']).size().reset_index(name='Total')

# Exibir o DataFrame agrupado
display(agrupados)

# Salvar o DataFrame agrupado em um novo arquivo Excel
output_file_path = os.path.join(caminho_arquivos, 'den_descartados_criterios_avaliacao.xlsx')
agrupados.to_excel(output_file_path, index=False)

In [ ]:
# Casos descartados, de acordo com o critério
den_descartados['CRITERIO'].value_counts().reset_index(name='Total')

In [ ]:
# Casos descartados, de acordo com o critério
den_descartados['CS_SEXO'].value_counts().reset_index(name='Total')

In [ ]:
# Casos descartados, de acordo com a zona de residência
den_descartados['CS_ZONA'].value_counts().reset_index(name='Total')

In [ ]:
df = pd.DataFrame(den_descartados)

# Agrupar por múltiplas colunas e contar o número de ocorrências
agrupados = df.groupby(['CRITERIO', 'DESCARTE_SEM_CRITERIO']).size().reset_index(name='Total')

# Exibir o DataFrame agrupado
display(agrupados)

In [ ]:
# Casos descartados, de acordo com o município de residência, por laboratório (10+)
file_path = 'descartados_2020_2024_lab.xlsx'
den_lab = pd.read_excel(file_path)

mun_10 = den_lab['ID_MN_RESI'].value_counts().reset_index(name='Total')

mun_10.head(10)

In [ ]:
# Casos descartados, de acordo com o município de residência, por clinico-epidemiologico (10+)
file_path = 'descartados_2020_2024_clepi.xlsx'
den_clepi = pd.read_excel(file_path)

mun_10 = den_clepi['ID_MN_RESI'].value_counts().reset_index(name='Total')

mun_10.head(10)

In [ ]:
# Converter colunas de datas para o tipo datetime
den_descartados['DT_NOTIFIC'] = pd.to_datetime(den_descartados['DT_NOTIFIC'], errors='coerce')
den_descartados['DT_ENCERRA'] = pd.to_datetime(den_descartados['DT_ENCERRA'], errors='coerce')

# Calcular a diferença de dias entre DT_NOTIFIC e DT_ENCERRA
den_descartados['DIF_DIAS'] = (den_descartados['DT_ENCERRA'] - den_descartados['DT_NOTIFIC']).dt.days

# Calcular a média das diferenças de dias
media_dif_dias = den_descartados['DIF_DIAS'].mean()

# Exibir a média
print(f'Tempo médio de encerramento do caso: {media_dif_dias:.2f} dias')

In [ ]:
print('Processo finalizado com sucesso!')
print('Avalie cuidadosamente os dados gerados, pois podem haver inconsistências/erros.')
print('Script generated by epiDEN Soluções Epidemiológicas')
print('Todos os direitos reservados')